# A graph build can fail before the declaration validator

Does a missing code always produce the same diagnostic during machine creation?

Run cells from top to bottom in a fresh Python kernel using the repository environment. Set the working directory to the repository root. This notebook is self-contained; do not run another declaration experiment in the same kernel. The matching `.py` file runs with `uv run python examples/step_21_relations/39_error_machine_build.py`.

The record holds common information; the pressing classes hold details specific to one kind of pressing. A `Specialization` link goes from the common record to those details, and a `Generalization` link points back. The `Classifier` marker declares which data code belongs to which kind. The [relations chapter](../../docs/tutorials/step-21-relations.md#generalization-and-specialization) explains the complete preparation.


## Imports

These imports supply the annotation tools and AOA types used below.

In [1]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseSpecEntity, Classifier, Generalization, Inverse, Rel, Specialization
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## MusicDomain

Group the music catalogue declarations. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [2]:
class MusicDomain(BaseDomain):
    """Group the music catalogue declarations."""

    name = "music"
    description = "A music catalogue"


## VinylRecordCoreEntity

Describe the information shared by all pressings. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [3]:
@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordCoreEntity(BaseCoreEntity):
    """Describe the information shared by all pressings."""

    id: str = Field(description="Record identifier")
    title: str = Field(description="Album title")
    media: str = Field(description="Pressing code")
    pressing: Annotated[
        Specialization[FirstPressSpecEntity | RepressSpecEntity | TestPressSpecEntity],
        Classifier(field="media", codes=Literal["early", "repress", "test"]),
        Inverse(field_name="record"),
    ] = Rel(description="Details of this pressing")


## FirstPressSpecEntity

Describe the stamper used for a first pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [4]:
@entity(description="First pressing", domain=MusicDomain)
class FirstPressSpecEntity(BaseSpecEntity):
    """Describe the stamper used for a first pressing."""

    id: str = Field(description="The head record identifier")
    stamper: str = Field(description="Stamper code")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier("record", Literal["first"]),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## RepressSpecEntity

Describe when an album was pressed again. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [5]:
@entity(description="Re-pressing", domain=MusicDomain)
class RepressSpecEntity(BaseSpecEntity):
    """Describe when an album was pressed again."""

    id: str = Field(description="The head record identifier")
    year: int = Field(description="Re-pressing year")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier("record", Literal["repress"]),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## TestPressSpecEntity

Describe who approved a test pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [6]:
@entity(description="Test pressing", domain=MusicDomain)
class TestPressSpecEntity(BaseSpecEntity):
    """Describe who approved a test pressing."""

    id: str = Field(description="The head record identifier")
    approved_by: str = Field(description="Reviewer name")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier("record", Literal["test"]),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [7]:
for model in (VinylRecordCoreEntity, FirstPressSpecEntity, RepressSpecEntity, TestPressSpecEntity):
    model.model_rebuild()


## The experiment

Does a missing code always produce the same diagnostic during machine creation?

In [8]:
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

try:
    ActionProductMachine(loggers=[])
except KeyError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The broken model built")


KeyError: 'early'


## Why this result occurs

The same declaration as example 23 reaches graph-edge construction during machine creation. That path tries to assemble labels before the declaration validator runs and raises KeyError for early. Calling the declaration validator directly, as the diagnostic examples do, provides the specific missing-code message. This is a current build-order limitation.